In [29]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

CLIENT_ID = os.getenv("DB_CLIENT_ID")
API_KEY = os.getenv("DB_API_KEY")

print("Client ID:", CLIENT_ID[:8] + "..." if CLIENT_ID else "MISSING")
print("API Key:", "loaded" if API_KEY else "MISSING")

Client ID: bb989c0e...
API Key: loaded


In [30]:
import requests

BASE_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/timetables/v1"

headers = {
    "DB-Client-Id": CLIENT_ID,
    "DB-Api-Key": API_KEY,
    "Accept": "application/xml",
}

# Step 1: find Berlin Hbf's eva number
response = requests.get(
    f"{BASE_URL}/station/BLS",   # BLS = official DS100 code for Berlin Hbf
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("First 500 chars:")
print(response.text[:500])

Status: 200
Content-Type: application/xml;charset=UTF-8
First 500 chars:
<stations>

<station p="11|12 D - G|12|13 A - D|13|14|13 D - G|13 A - C|13 C - D|11 D - G|14 A - D|14 A - C|14 C - D|14 E - F|11 C - D|13 E - F|11 E - F|11 A - D|12 A - D|14 D - G|12 C - D|12 E - F" meta="8070952|8089021|8098160" name="Berlin Hbf" eva="8011160" ds100="BLS" db="true" creationts="26-05-08 10:15:24.735"/>

</stations>



In [31]:
from datetime import datetime

EVA_BERLIN_HBF = "8011160"

now = datetime.now()
date_str = now.strftime("%y%m%d")   # e.g. "260515"
hour_str = now.strftime("%H")       # e.g. "14"

print(f"Fetching planned timetable for date={date_str} hour={hour_str}")

response = requests.get(
    f"{BASE_URL}/plan/{EVA_BERLIN_HBF}/{date_str}/{hour_str}",
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Length:", len(response.text), "chars")
print("First 1500 chars:")
print(response.text[:1500])

Fetching planned timetable for date=260515 hour=20


Status: 200
Length: 7396 chars
First 1500 chars:
<?xml version='1.0' encoding='UTF-8'?><timetable station='Berlin Hbf'><s id="-5637803255857346220-2605151528-11"><tl f="F" t="p" o="80" c="ICE" n="943"/><ar pt="2605152016" pp="12" fb="ICE 943" ppth="Köln Hbf|Düsseldorf Hbf|Düsseldorf Flughafen|Duisburg Hbf|Essen Hbf|Bochum Hbf|Dortmund Hbf|Hamm(Westf)Hbf|Hannover Hbf|Wolfsburg Hbf"/><dp pt="2605152020" pp="12" fb="ICE 943" hi="1" ppth="Berlin Ostbahnhof"/></s><s id="-8301258216865380478-2605151912-16"><tl f="D" t="p" o="OWRE" c="OE" n="73743"/><ar pt="2605152057" pp="11" l="RE1" fb="RE1" ppth="Magdeburg Hbf|Magdeburg-Neustadt|Burg(Magdeburg)|Güsen(b Genthin)|Genthin|Wusterwitz|Kirchmöser|Brandenburg Hbf|Götz|Groß Kreutz|Werder(Havel)|Potsdam Hbf|Berlin-Wannsee|Berlin-Charlottenburg|Berlin Zoologischer Garten"/><dp pt="2605152059" pp="11" l="RE1" fb="RE1" ppth="Berlin Friedrichstraße|Berlin Alexanderplatz|Berlin Ostbahnhof|Berlin Ostkreuz|Erkner|Fangschleuse|Hangelsberg|Fürstenwalde(Spre

In [32]:
response = requests.get(
    f"{BASE_URL}/fchg/{EVA_BERLIN_HBF}",
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Length:", len(response.text), "chars")
print("First 1500 chars:")
print(response.text[:1500])

Status: 200
Length: 135536 chars
First 1500 chars:
<timetable station="Berlin Hbf" eva="8011160">

<s id="738303672697175305-2605151327-11" eva="8011160">
    <m id="r2621044" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605151108" ts-tts="26-05-15 11:08:21.638" pr="2"/>
    <m id="r2621110" t="h" from="2605151300" to="2605151500" cat="Störung" ts="2605151351" ts-tts="26-05-15 13:51:21.250" pr="1"/>
    <m id="r2621151" t="h" from="2605151400" to="2605151600" cat="Störung" ts="2605151500" ts-tts="26-05-15 15:00:48.559" pr="1"/>
    <m id="r2621064" t="h" from="2605151100" to="2605151500" cat="Störung" ts="2605151200" ts-tts="26-05-15 14:16:04.056" pr="1"/>
    <m id="r2620992" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605150840" ts-tts="26-05-15 08:40:58.275" pr="2"/>
    <ar ct="2605151818">
        <m id="r56582449" t="q" c="92" ts="2605151108" ts-tts="26-05-15 11:08:37.338"/>
        <m id="r56582452" t="q" c="92" ts="2605151108" ts-tts="26-05-

In [33]:
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime

def parse_db_time(s):
    """DB times are YYMMDDHHMM strings, e.g. '2605151644' = 2026-05-15 16:44"""
    if not s:
        return None
    return datetime.strptime(s, "%y%m%d%H%M")

def parse_plan_xml(xml_text):
    """Parse a /plan/... response into a list of dicts."""
    root = ET.fromstring(xml_text)
    rows = []
    for s in root.findall("s"):
        stop_id = s.get("id")
        tl = s.find("tl")
        ar = s.find("ar")
        dp = s.find("dp")

        row = {
            "stop_id": stop_id,
            "train_category": tl.get("c") if tl is not None else None,
            "train_number": tl.get("n") if tl is not None else None,
            "operator": tl.get("o") if tl is not None else None,
            "planned_arrival": parse_db_time(ar.get("pt")) if ar is not None else None,
            "planned_arrival_platform": ar.get("pp") if ar is not None else None,
            "arrival_line": ar.get("l") if ar is not None else None,
            "arrival_path": ar.get("ppth") if ar is not None else None,
            "planned_departure": parse_db_time(dp.get("pt")) if dp is not None else None,
            "planned_departure_platform": dp.get("pp") if dp is not None else None,
            "departure_line": dp.get("l") if dp is not None else None,
            "departure_path": dp.get("ppth") if dp is not None else None,
        }
        rows.append(row)
    return rows


# Re-fetch the plan and parse
date_str = datetime.now().strftime("%y%m%d")
hour_str = datetime.now().strftime("%H")

plan_response = requests.get(
    f"{BASE_URL}/plan/{EVA_BERLIN_HBF}/{date_str}/{hour_str}",
    headers=headers,
    timeout=30,
)

plan_rows = parse_plan_xml(plan_response.text)
plan_df = pd.DataFrame(plan_rows)

print(f"Got {len(plan_df)} scheduled stops at Berlin Hbf for hour {hour_str}")
plan_df.head(10)

Got 16 scheduled stops at Berlin Hbf for hour 20


,stop_id,train_category,train_number,operator,planned_arrival,planned_arrival_platform,arrival_line,arrival_path,planned_departure,planned_departure_platform,departure_line,departure_path
0,-5637803255857346220-2605151528-11,ICE,943,80,2026-05-15 20:16:00,12,NaN,Köln Hbf|Düsseldorf Hbf|Düsseldorf Flughafen|D...,2026-05-15 20:20:00,12,NaN,Berlin Ostbahnhof
1,-8301258216865380478-2605151912-16,OE,73743,OWRE,2026-05-15 20:57:00,11,RE1,Magdeburg Hbf|Magdeburg-Neustadt|Burg(Magdebur...,2026-05-15 20:59:00,11,RE1,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
2,-6340897605174334619-2605152014-2,ICE,679,80,2026-05-15 20:25:00,13,NaN,Berlin Ostbahnhof,2026-05-15 20:29:00,13,NaN,Berlin Zoologischer Garten|Berlin-Spandau|Wolf...
3,-2162136752730629646-2605151956-2,ICE,740,80,2026-05-15 20:03:00,13,NaN,Berlin Ostbahnhof,2026-05-15 20:07:00,13,NaN,Berlin-Spandau|Hannover Hbf|Bünde(Westf)|Osnab...
4,265997371841578616-2605151944-9,OE,73783,OWRE,2026-05-15 20:35:00,12,RE1,Brandenburg Hbf|Werder(Havel)|Potsdam Park San...,2026-05-15 20:37:00,12,RE1,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
5,7996741227960904838-2605151705-20,RSM,80429,RSMD,2026-05-15 20:47:00,11,HBX,Goslar|Vienenburg|Stapelburg|Ilsenburg|Darling...,2026-05-15 20:49:00,11,HBX,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
6,2869769973922245820-2605152020-2,ICE,540,80,2026-05-15 20:29:00,14,NaN,Berlin Ostbahnhof,2026-05-15 20:33:00,14,NaN,Berlin Zoologischer Garten|Berlin-Spandau|Sten...
7,-3637498970395766319-2605151907-8,OE,73788,OWRE,2026-05-15 20:18:00,14,RE1,Frankfurt(Oder)|Fürstenwalde(Spree)|Erkner|Ber...,2026-05-15 20:20:00,14,RE1,Berlin Zoologischer Garten|Berlin-Charlottenbu...
8,-1496207040924626953-2605151838-15,OE,73746,OWRE,2026-05-15 19:59:00,14,RE1,Frankfurt(Oder)|Frankfurt(Oder)-Rosengarten|Pi...,2026-05-15 20:01:00,14,RE1,Berlin Zoologischer Garten|Berlin-Charlottenbu...
9,8082573176163790166-2605151935-6,RE,3132,800165,2026-05-15 20:24:00,11,RE2,Hennigsdorf(b Berlin)|Dallgow-Döberitz|Berlin-...,2026-05-15 20:26:00,11,RE2,Berlin Friedrichstraße|Berlin Alexanderplatz|B...


In [34]:
fchg_response = requests.get(
    f"{BASE_URL}/fchg/{EVA_BERLIN_HBF}",
    headers=headers,
    timeout=30,
)

print("Status:", fchg_response.status_code)
print("Length:", len(fchg_response.text), "chars")
print("First 2000 chars:")
print(fchg_response.text[:2000])

Status: 200
Length: 135536 chars
First 2000 chars:
<timetable station="Berlin Hbf" eva="8011160">

<s id="738303672697175305-2605151327-11" eva="8011160">
    <m id="r2621044" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605151108" ts-tts="26-05-15 11:08:21.638" pr="2"/>
    <m id="r2621110" t="h" from="2605151300" to="2605151500" cat="Störung" ts="2605151351" ts-tts="26-05-15 13:51:21.250" pr="1"/>
    <m id="r2621151" t="h" from="2605151400" to="2605151600" cat="Störung" ts="2605151500" ts-tts="26-05-15 15:00:48.559" pr="1"/>
    <m id="r2621064" t="h" from="2605151100" to="2605151500" cat="Störung" ts="2605151200" ts-tts="26-05-15 14:16:04.056" pr="1"/>
    <m id="r2620992" t="h" from="2605151327" to="2605151829" cat="Information" ts="2605150840" ts-tts="26-05-15 08:40:58.275" pr="2"/>
    <ar ct="2605151818">
        <m id="r56582449" t="q" c="92" ts="2605151108" ts-tts="26-05-15 11:08:37.338"/>
        <m id="r56582452" t="q" c="92" ts="2605151108" ts-tts="26-05-

In [35]:
def parse_fchg_xml(xml_text):
    """Parse a /fchg/... response into a list of dicts."""
    root = ET.fromstring(xml_text)
    rows = []
    for s in root.findall("s"):
        stop_id = s.get("id")
        ar = s.find("ar")
        dp = s.find("dp")

        # collect delay reason codes
        reasons = []
        for m in s.iter("m"):
            t = m.get("t")
            c = m.get("c")
            if t == "d" and c:   # delay code
                reasons.append(c)

        row = {
            "stop_id": stop_id,
            "changed_arrival": parse_db_time(ar.get("ct")) if ar is not None and ar.get("ct") else None,
            "changed_arrival_platform": ar.get("cp") if ar is not None else None,
            "changed_departure": parse_db_time(dp.get("ct")) if dp is not None and dp.get("ct") else None,
            "changed_departure_platform": dp.get("cp") if dp is not None else None,
            "delay_codes": ",".join(sorted(set(reasons))) if reasons else None,
        }
        rows.append(row)
    return rows


fchg_rows = parse_fchg_xml(fchg_response.text)
fchg_df = pd.DataFrame(fchg_rows)

print(f"Got {len(fchg_df)} changed stops")
fchg_df.head(10)

Got 232 changed stops


,stop_id,changed_arrival,changed_arrival_platform,changed_departure,changed_departure_platform,delay_codes
0,738303672697175305-2605151327-11,2026-05-15 18:18:00,NaN,2026-05-15 18:22:00,NaN,NaN
1,-7486970227320284570-2605151426-14,2026-05-15 19:48:00,NaN,2026-05-15 19:52:00,NaN,7
2,-359164097413752423-2605151522-7,2026-05-15 16:14:00,11,2026-05-15 16:16:00,11,NaN
3,9074417543825272152-2605151544-9,2026-05-15 16:35:00,NaN,2026-05-15 16:37:00,NaN,21
4,1144018718150522635-2605152012-16,2026-05-15 21:57:00,NaN,2026-05-15 21:59:00,NaN,NaN
5,1142963549373501064-2605151128-11,2026-05-15 17:01:00,NaN,2026-05-15 17:05:00,NaN,"3,34"
6,6637172419819473914-2605151040-15,2026-05-15 16:51:00,NaN,2026-05-15 16:54:00,NaN,2
7,-8982296658299132323-2605151505-19,2026-05-15 16:56:00,NaN,2026-05-15 16:57:00,NaN,64
8,-8315914047777309242-2605151200-13,2026-05-15 17:49:00,NaN,2026-05-15 17:55:00,NaN,NaN
9,3410312917486180766-2605151641-6,2026-05-15 17:08:00,NaN,2026-05-15 17:10:00,NaN,NaN


In [36]:
# Join planned with changes by stop_id
merged = plan_df.merge(fchg_df, on="stop_id", how="left")

# Compute delays in minutes
merged["arrival_delay_min"] = (
    (merged["changed_arrival"] - merged["planned_arrival"]).dt.total_seconds() / 60
)
merged["departure_delay_min"] = (
    (merged["changed_departure"] - merged["planned_departure"]).dt.total_seconds() / 60
)

# Sort by departure delay, biggest first
delayed = (
    merged[merged["departure_delay_min"].notna() & (merged["departure_delay_min"] != 0)]
    .sort_values("departure_delay_min", ascending=False)
    [["train_category", "train_number", "departure_line",
      "planned_departure", "changed_departure", "departure_delay_min",
      "delay_codes"]]
)

print(f"{len(delayed)} delayed trains right now")
delayed.head(20)

1 delayed trains right now


,train_category,train_number,departure_line,planned_departure,changed_departure,departure_delay_min,delay_codes
0,ICE,943,NaN,2026-05-15 20:20:00,2026-05-15 20:27:00,7.0,"48,7"


In [37]:
from datetime import timedelta

def fetch_plan_window(eva, hours_ahead=6):
    """Fetch /plan for the current hour + the next N hours."""
    all_rows = []
    now = datetime.now()
    for offset in range(hours_ahead + 1):
        t = now + timedelta(hours=offset)
        date_str = t.strftime("%y%m%d")
        hour_str = t.strftime("%H")
        r = requests.get(
            f"{BASE_URL}/plan/{eva}/{date_str}/{hour_str}",
            headers=headers,
            timeout=30,
        )
        if r.status_code == 200:
            all_rows.extend(parse_plan_xml(r.text))
        else:
            print(f"Hour {hour_str}: status {r.status_code}")
    return all_rows


plan_rows = fetch_plan_window(EVA_BERLIN_HBF, hours_ahead=6)
plan_df = pd.DataFrame(plan_rows)
print(f"Got {len(plan_df)} planned stops over the next 7 hours")

Got 64 planned stops over the next 7 hours


In [38]:
merged = plan_df.merge(fchg_df, on="stop_id", how="left")
merged["departure_delay_min"] = (
    (merged["changed_departure"] - merged["planned_departure"]).dt.total_seconds() / 60
)

delayed = (
    merged[merged["departure_delay_min"].notna() & (merged["departure_delay_min"] != 0)]
    .sort_values("departure_delay_min", ascending=False)
    [["train_category", "train_number", "departure_line",
      "planned_departure", "changed_departure", "departure_delay_min",
      "delay_codes"]]
)

print(f"{len(delayed)} delayed trains")
delayed.head(20)

1 delayed trains


,train_category,train_number,departure_line,planned_departure,changed_departure,departure_delay_min,delay_codes
0,ICE,943,NaN,2026-05-15 20:20:00,2026-05-15 20:27:00,7.0,"48,7"
